In [1]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# 1. Load the pre-trained Encoder-Decoder model and Tokenizer
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# 2. Define the input sequence
input_text = "translate English to French: The encoder and decoder architecture is very powerful."
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

print("--- STANDARD END-TO-END GENERATION ---")
# The model internally passes data through the encoder, gets the context vector,
# and autoregressively generates output using the decoder.
outputs = model.generate(input_ids, max_length=20)
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Input:  {input_text}")
print(f"Output: {decoded_output}\n")

print("--- INSIDE THE ARCHITECTURE (STEP-BY-STEP) ---")

# STEP A: THE ENCODER
# We pass the input IDs strictly into the encoder.
# It surveys the entire sentence and outputs the "Context Vector" (hidden states).
encoder_outputs = model.encoder(input_ids=input_ids)
context_vector = encoder_outputs.last_hidden_state

print(f"Context Vector Shape: {context_vector.shape}")
# Shape will be [Batch Size, Sequence Length, Hidden Dimension]
print("The encoder has successfully mapped the input into a mathematical representation.\n")

# STEP B: THE DECODER
# To start decoding, the decoder needs a starting token (usually a padding or Start-Of-Sequence token).
decoder_input_ids = tokenizer("<pad>", return_tensors="pt", add_special_tokens=False).input_ids

# We pass the starting token AND the encoder's context vector into the decoder.
decoder_outputs = model.decoder(
    input_ids=decoder_input_ids,
    encoder_hidden_states=context_vector
)

# The decoder looks at the context vector and predicts the mathematical state for the first translated word.
first_word_hidden_state = decoder_outputs.last_hidden_state

print(f"Decoder Output Shape (First token): {first_word_hidden_state.shape}")
print("The decoder uses the context vector and autoregressive masking to generate the translation token by token.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

--- STANDARD END-TO-END GENERATION ---
Input:  translate English to French: The encoder and decoder architecture is very powerful.
Output: L'architecture de codeur et de décodeur est très puissante.

--- INSIDE THE ARCHITECTURE (STEP-BY-STEP) ---
Context Vector Shape: torch.Size([1, 18, 512])
The encoder has successfully mapped the input into a mathematical representation.

Decoder Output Shape (First token): torch.Size([1, 1, 512])
The decoder uses the context vector and autoregressive masking to generate the translation token by token.
